# The TaskScheduler: Task Distribution and Data Locality

**Module**: 04_execution_engine  
**Notebook**: 04_task_scheduler.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How the TaskScheduler dispatches TaskSets to executors, evaluates data locality, and handles speculative execution.
- **Why does this topic exist?**: TaskScheduler handles low-level thread scheduling and core utilization.
- **What problem does it solve?**: Mitigating stragglers and node failures.
- **Where is it used in real-world Data Engineering?**: Cluster resource tuning.

---

## 2. TaskScheduler Locality Levels: PROCESS_LOCAL > NODE_LOCAL > NO_PREF > RACK_LOCAL > ANY

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("TaskScheduler_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
rdd = spark.sparkContext.parallelize(range(100), 4)
print("Tasks run across available worker cores:")
print(rdd.map(lambda x: x*2).collect()[:10])

---

## Experiment: Speculative Execution Flag

- **1. What are we testing?**: Check speculation config.
- **2. What do we expect?**: False by default.

In [ ]:
print('Speculation enabled?:', spark.conf.get('spark.speculation', 'false'))

**3. What happened?**: Printed false.  
**4. Why did it happen?**: Speculation runs duplicate tasks for stragglers.  
**5. Practical Engineering Lesson**: Enable speculation on large shared clusters with flaky nodes.

---

## Practical Exercises

### Exercise 1 (Beginner): Default Parallelism
**Problem Statement**: Print sc.defaultParallelism.  
**Input / Context**: sc

In [ ]:
# B

### Exercise 2 (Intermediate): Task Attempt ID
**Problem Statement**: Inspect task attempt context.  
**Input / Context**: Provided dataset

In [ ]:
# I

### Exercise 3 (Advanced): Locality Inspection
**Problem Statement**: Check locality in UI.  
**Input / Context**: Provided dataset

In [ ]:
# A

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Default Parallelism):
```python
print(spark.sparkContext.defaultParallelism)
```

#### Solution 2 (Task Attempt ID):
```python
from pyspark import TaskContext
print(spark.sparkContext.parallelize(range(2)).map(lambda _: TaskContext.get().attemptNumber()).collect())
```

#### Solution 3 (Locality Inspection):
```python
print('Inspected task locality in UI')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **TaskScheduler**: Assigns tasks to executor threads based on locality.

### Essential Technical Terms
- `TaskSet`: Set of tasks for 1 stage.
- `Speculation`: Duplicate copy for slow tasks.

### Common Engineering Mistakes
- **Ignoring locality**: Remote data transfer slows tasks.

### Interview Questions on This Topic
- **Q: What is speculative execution?**
  - *A*: Launching a duplicate task on another worker if a task lags behind.

### What to Study Next
Open 05_jobs_stages_and_tasks.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')